## Custom SNN Python implementation for electronics engineering bachelor project

In [141]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set random seed for reproducability
np.random.seed(42)

### Leaky integrate and fire neuron

In [142]:
class LIF():
    """
        Simple Leaky Integrate-and-Fire (LIF) Neuron with STDP
    """
    def __init__(self, beta=0.9, threshold=2.0, reset=0.0, learning_rate=0.01):
        self.beta = beta                    # Decay factor
        self.threshold = threshold          # Spike threshold value - value that triggers a spike
        self.mem = 0                        # Membrane potential - current value
        self.reset = reset                  # Membrane potential reset value - value after spike
        self.last_spike = -1                # Time of last spike - counts time steps since last spike (-1 = no spike yet)
        self.learning_rate = learning_rate  # Learning rate for STDP
        self.spk = 0                        # Spike
        self.spikeWindow = 5                # Spikes need to be within 5 timesteps of eachother to trigger weight change
        

    def update(self, synaptic_input):
            """
            Update membrane potential and check if a spike should be generated
            """
            # Reset spike from previous timestep
            self.spk = 0
            
            # Increment time since last spike
            if self.last_spike >= 0:
                self.last_spike += 1
            
            # Compute decay and membrane potential
            decay = self.beta * self.mem
            self.mem = decay + synaptic_input

            
            # If membrane potential exceeds threshold -> spike
            if self.mem > self.threshold:
                self.spk = 1
                self.last_spike = 0  # Reset last spike time
                self.mem = self.reset
            
            return self.spk
    
    def update_threshold(self, baseline=1.5, step=0.05, threshold_decay=0.01):
        """ 
        Increase threshold on spike, slowly decay to baseline threshold otherwise.
        """
        # Increase threshold if spike
        if self.spk:
            self.threshold += step
        else:
            # Gradually move back down to the baseline
            if self.threshold > baseline:
                self.threshold -= threshold_decay

    def STDP(self, weight, pre_spike_time):
        """
        Basic Spike-Timing-Dependent Plasticity (STDP).
        """
        # Potentiation: The output (post) neuron just fired
        if self.last_spike == 0:
            if 0 < pre_spike_time <= self.spikeWindow:  # Pre fired before post and within the window
                weight += self.learning_rate
                
        # Depression: The input (pre) neuron just fired
        if pre_spike_time == 0:
            if 0 < self.last_spike <= self.spikeWindow: # Pre fired after post and within the window
                weight -= self.learning_rate
                
        return np.clip(weight, 0.001, 1.0)
    
    def rSTDP(self, weight, prespike, reward):
        # STDP with reward modulation
        pass


In [143]:
# Simulation parameters
time_steps = 300

# Architecture parameter
input_size = 25     # Number of input neurons
output_size = 3     # Number of output neurons

# Hyperparameters
threshold = 1.5
learning_rate = 0.1
initial_weight = 0.5

# Instantiate synapses as a weight matrix, with all weights starting at
input_to_output_synapses = np.full((input_size, output_size), 0.1)
#input_to_output_synapses = np.random.rand(input_size, output_size)
print(f"Sucessfully created synapse matrix {input_to_output_synapses.shape}")

# Instantiate layers as litst of neurons
input_neurons = [LIF(beta=0.9, threshold=threshold, learning_rate=learning_rate) for _ in range(input_size)]
output_neurons = [LIF(beta=0.9, threshold=threshold, learning_rate=learning_rate) for _ in range(output_size)]

print(f"Sucessfully created input layer with {len(input_neurons)}")
print(f"Sucessfully created output layer with {len(output_neurons)}")




Sucessfully created synapse matrix (25, 3)
Sucessfully created input layer with 25
Sucessfully created output layer with 3


In [ ]:
# Pattern: Corner top left
pattern_top_left_corner = np.array([
    1, 1, 1, 1, 1,
    1, 0, 0, 0, 0,
    1, 0, 0, 0, 0,
    1, 0, 0, 0, 0,
    1, 0, 0, 0, 0
])
# Pattern: Corner top right
pattern_top_right_corner = np.array([
    0, 1, 1, 1, 1,
    0, 0, 0, 0, 1,
    0, 0, 0, 0, 1,
    0, 0, 0, 0, 1,
    0, 0, 0, 0, 1
])
# Pattern: Border pixels only
pattern_square = np.array([
    1, 1, 1, 1, 1,
    1, 0, 0, 0, 1,
    1, 0, 0, 0, 1,
    1, 0, 0, 0, 1,
    1, 1, 1, 1, 1
])
# Pattern: Diagonal line
pattern_diagonal = np.array([
    1, 0, 0, 0, 0,
    0, 1, 0, 0, 0,
    0, 0, 1, 0, 0,
    0, 0, 0, 1, 0,
    0, 0, 0, 0, 1
])

In [ ]:
# Simulation
for t in range(time_steps):
    # Hold each pattern for 10 timesteps to allow integration
    if (t // 10) % 3 == 0:
        input_spikes = pattern_top_left_corner
    elif (t // 10) % 3 == 1:
        input_spikes = pattern_square
    else:
        input_spikes = pattern_diagonal



    # Update input layer neurons
    for i in range(input_size):
        input_neurons[i].update(input_spikes[i])

    # Capture weights before update for diagnostic consistency
    current_weights = input_to_output_synapses.copy()

    # Update output layer neurons
    for j in range(output_size):
        # Compute total synaptic input using input neuron spike states
        synaptic_input = sum(current_weights[i,j] * input_neurons[i].spk for i in range(input_size))
        output_neurons[j].update(synaptic_input)

    # Only let one output neuron fire at a time, inhibit theothers
    # Check which neuron(s) spiked
    fired_indices = [j for j, n in enumerate(output_neurons) if n.spk]
    if len(fired_indices) > 0:
        winner = np.random.choice(fired_indices)  # Choose a random winner if multiple spikes at the current timestep
        for j in range(output_size):
            if j != winner:
                output_neurons[j].mem = 0
                output_neurons[j].spk = 0

    # Apply STDP learning rule
    for i in range(input_size):
        for j in range(output_size):
            input_to_output_synapses[i,j] = output_neurons[j].STDP(input_to_output_synapses[i,j], input_neurons[i].last_spike)

    
    # Update thresholds based on spikes
    for j in range(output_size):
        output_neurons[j].update_threshold()

    # Identify which output neurons fired
    fired_indices = [j for j, neuron in enumerate(output_neurons) if neuron.spk]

    # Print diagnostics at current timestep
    input_spike_count = sum(neuron.spk for neuron in input_neurons)
    output_spike_count = len(fired_indices)

    # Get membrane potentials for all output neurons
    mem_potentials = [f"{neuron.mem:.2f}" for neuron in output_neurons]

    # Calculate total synaptic input received by first output neuron (using pre-update weights)
    syn_in_diag = sum(current_weights[i,0] * input_neurons[i].spk for i in range(input_size))

    # Get spike timing info for first output neuron
    last_spike_time = output_neurons[0].last_spike

    print(f"\n=== t={t} ===")
    print(f"Input spikes: {input_spike_count}/{input_size}")
    print(f"Output spikes: {output_spike_count}/{output_size} (Indices: {fired_indices})")
    print(f"Output membrane potentials: [{', '.join(mem_potentials)}]")
    print(f"Synaptic input (neuron 0): {syn_in_diag:.4f}")
    print(f"Last spike time (neuron 0): {last_spike_time}")
    print(f"Weight range: [{input_to_output_synapses.min():.3f}, {input_to_output_synapses.max():.3f}]")
    print(f"Weight mean: {input_to_output_synapses.mean():.3f}")


=== t=0 ===
Input spikes: 0/25
Output spikes: 0/3 (Indices: [])
Output membrane potentials: [0.00, 0.00, 0.00]
Synaptic input (neuron 0): 0.0000
Last spike time (neuron 0): -1
Weight range: [0.100, 0.100]
Weight mean: 0.100

=== t=1 ===
Input spikes: 9/25
Output spikes: 0/3 (Indices: [])
Output membrane potentials: [0.90, 0.90, 0.90]
Synaptic input (neuron 0): 0.9000
Last spike time (neuron 0): -1
Weight range: [0.100, 0.100]
Weight mean: 0.100

=== t=2 ===
Input spikes: 0/25
Output spikes: 0/3 (Indices: [])
Output membrane potentials: [0.81, 0.81, 0.81]
Synaptic input (neuron 0): 0.0000
Last spike time (neuron 0): -1
Weight range: [0.100, 0.100]
Weight mean: 0.100

=== t=3 ===
Input spikes: 9/25
Output spikes: 1/3 (Indices: [2])
Output membrane potentials: [0.00, 0.00, 0.00]
Synaptic input (neuron 0): 0.9000
Last spike time (neuron 0): 0
Weight range: [0.100, 0.100]
Weight mean: 0.100

=== t=4 ===
Input spikes: 0/25
Output spikes: 0/3 (Indices: [])
Output membrane potentials: [0.00, 

In [146]:
def visualize_weights(synapse_matrix, output_idx):
    # Reshape the weights for one specific output neuron to 5x5
    weights = synapse_matrix[:, output_idx].reshape(5, 5)
    print(f"\nWeights for Neuron {output_idx}:")
    for row in weights:
        # Print a '#' for strong weights, '.' for weak weights
        print(" ".join(["#" if w > 0.5 else "." for w in row]))

# Example use after simulation:
visualize_weights(input_to_output_synapses, 0)


Weights for Neuron 0:
. . . . .
. . . . .
. . . . .
. . . . .
. . . . .
